# 阶段 6：XGBoost 截面预测

本 Notebook 读取 `config/xgboost.yml` 中的阶段 6 产物，检查 RL 公式因子宽表、逐月 walk-forward 预测、IC/RankIC 和 feature importance。这里不做组合回测，也不引入交易约束。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from quant_rl_alpha.utils.config import load_config
from quant_rl_alpha.utils.paths import project_root

config = load_config("xgboost")
root = project_root()

paths = {name: root / path for name, path in config["outputs"].items()}
paths

## 数据集覆盖

In [ ]:
dataset = pd.read_parquet(paths["dataset"])
feature_cols = [col for col in dataset.columns if col.startswith("alpha_")]

summary = pd.DataFrame(
    {
        "rows": [len(dataset)],
        "months": [dataset["date"].nunique()],
        "symbols": [dataset["symbol"].nunique()],
        "features": [len(feature_cols)],
        "start": [dataset["date"].min()],
        "end": [dataset["date"].max()],
    }
)
summary

In [ ]:
coverage = dataset.groupby("date")[feature_cols].apply(lambda frame: frame.notna().mean().mean())
coverage.plot(title="Feature coverage by prediction month", figsize=(10, 3))
plt.ylabel("coverage")
plt.show()

## Walk-forward 预测指标

In [ ]:
predictions = pd.read_parquet(paths["predictions"])
metrics = pd.read_parquet(paths["metrics"])
metrics.tail()

In [ ]:
ax = metrics.set_index("date")[["ic", "rank_ic"]].plot(figsize=(10, 4), marker="o")
ax.axhline(0, color="black", linewidth=1)
plt.title("Monthly prediction IC / RankIC")
plt.show()

metrics[["ic", "rank_ic", "feature_coverage"]].describe()

## 预测分数与分组收益

In [ ]:
predictions["score"].hist(bins=50, figsize=(8, 3))
plt.title("Prediction score distribution")
plt.show()

In [ ]:
grouped = predictions.copy()
grouped["score_decile"] = grouped.groupby("date")["score"].transform(
    lambda item: pd.qcut(item.rank(method="first"), 10, labels=False, duplicates="drop") + 1
)
decile_return = grouped.groupby("score_decile")["future_20d_return"].mean()
decile_return.plot(kind="bar", title="Mean future 20d return by score decile", figsize=(8, 3))
plt.ylabel("mean future_20d_return")
plt.show()
decile_return

## Feature importance

In [ ]:
importance = pd.read_parquet(paths["feature_importance"])
importance.head(15)

In [ ]:
top_importance = importance.sort_values("gain", ascending=False).head(15)
top_importance.plot.barh(x="feature", y="gain", figsize=(8, 5), legend=False)
plt.gca().invert_yaxis()
plt.title("Top feature importance by average gain")
plt.show()